# Day 2 — Convert to JPG subset + extract Impression + clean dataset

**Goal:** map DICOM ids to JPGs (MIMIC-CXR-JPG), download only matching JPGs, extract `IMPRESSION` section, and save a clean dataset CSV.

In [ ]:
!pip -q install pandas tqdm

In [ ]:

# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, math, random, time
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/mimic_project"
RAW  = f"{BASE}/raw"
DATA = f"{BASE}/dataset"
IMAGES = f"{DATA}/images"
REPORTS_DIR = f"{DATA}/reports"

os.makedirs(RAW, exist_ok=True)
os.makedirs(IMAGES, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("BASE:", BASE)
print("RAW:", RAW)
print("IMAGES:", IMAGES)
print("REPORTS_DIR:", REPORTS_DIR)


## 1) Download MIMIC-CXR-JPG metadata + labels (small files)

In [ ]:

USER = "YOUR_PHYSIONET_USERNAME"  # <-- change
!wget -N --user {USER} --ask-password -P {RAW} https://physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-metadata.csv.gz
!wget -N --user {USER} --ask-password -P {RAW} https://physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-chexpert.csv.gz
!ls -lh {RAW} | head


## 2) Load Day 1 manifest and join to JPG metadata

In [ ]:

manifest = pd.read_csv(f"{BASE}/manifest_with_reports_2000studies.csv")
jpg_meta = pd.read_csv(f"{RAW}/mimic-cxr-2.0.0-metadata.csv.gz")

# Keep only dicom_id keys + view position + subject/study
keep_cols = ["dicom_id","subject_id","study_id","ViewPosition"]
jpg_meta = jpg_meta[keep_cols].copy()

df = manifest.merge(jpg_meta, on=["dicom_id","subject_id","study_id"], how="left")
print("Rows after merge:", len(df))
print("Missing ViewPosition:", df["ViewPosition"].isna().sum())

# Drop rows not available in JPG release
df = df.dropna(subset=["ViewPosition"]).copy()
print("Rows with JPG available:", len(df))


## 3) Create download URLs and download only those JPGs

In [ ]:

# The 'path' in manifest is like: files/p10/p10002930/s55885481/<dicom_id>.dcm
# JPG path is same but .jpg
def dcm_to_jpg_relpath(p):
    return p.replace(".dcm", ".jpg")

df["jpg_relpath"] = df["path"].apply(dcm_to_jpg_relpath)
base_url = "https://physionet.org/files/mimic-cxr-jpg/2.0.0/"

url_list_path = f"{BASE}/download_jpg_urls.txt"
with open(url_list_path, "w") as f:
    for rel in df["jpg_relpath"]:
        f.write(base_url + rel + "\n")

print("Saved download list:", url_list_path)
print("Example URL:", base_url + df["jpg_relpath"].iloc[0])

# Download (credentialed)
USER = "YOUR_PHYSIONET_USERNAME"  # <-- change
!wget -c -N --user {USER} --ask-password -i {url_list_path} -P {IMAGES}


## 4) Extract Impression section and build final clean dataset

In [ ]:

def extract_impression(report_text: str) -> str:
    if not isinstance(report_text, str) or not report_text.strip():
        return ""
    # Normalize
    t = report_text.replace("\r\n", "\n").replace("\r", "\n")
    # Find "IMPRESSION:" block if present
    m = re.search(r"(?is)\bIMPRESSION\s*:\s*(.*)$", t)
    if m:
        imp = m.group(1).strip()
    else:
        # fallback: keep last paragraph-ish
        parts = [p.strip() for p in t.split("\n\n") if p.strip()]
        imp = parts[-1] if parts else ""
    # Trim very long
    return imp[:2000]

df["impression"] = df["report_text"].apply(extract_impression)
before = len(df)
df = df[df["impression"].str.len() > 0].copy()
print("Before:", before, "After keeping impressions:", len(df))

# Save image_path
df["image_path"] = df["dicom_id"].astype(str) + ".jpg"
df["image_path"] = df["image_path"].apply(lambda x: os.path.join(IMAGES, x))

# Verify images exist
exists = df["image_path"].apply(os.path.exists)
print("Images found:", int(exists.sum()))
print("Missing images:", int((~exists).sum()))
df = df[exists].copy()

clean = df[["subject_id","study_id","dicom_id","ViewPosition","image_path","impression"]].copy()
out_csv = f"{BASE}/clean_dataset_2696.csv"
clean.to_csv(out_csv, index=False)
print("Saved:", out_csv, "Rows:", len(clean))
clean.head()


✅ **Day 2 deliverable:** `clean_dataset_2696.csv` + downloaded JPGs under `dataset/images/`